In [ ]:
%pip install --upgrade pip
%pip uninstall -y pinecone-client
%pip install --prefer-binary pinecone

import os
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
import itertools
import random
import pandas as pd
import numpy as np
from uuid import uuid4


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


False

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

client = OpenAI(api_key=openai_api_key)
pc = Pinecone(api_key=pinecone_api_key)

In [6]:
index_name = "pc-3"

In [ ]:
index = pc.Index(index_name, pool_threads = 10)

In [8]:
def retrieve_docs(query):
    query_embedding = client.embeddings.create( input = query, model = "text-embedding-3-large").data[0].embedding
    results = index.query( vector = query_embedding, top_k = 3, include_metadata = True, namespace = "squad_dataset")
    
    retrieve_docs = []
    sources = []
    
    for match in results["matches"]:
        retrieve_docs.append(match["metadata"]["text"])
        sources.append(match["metadata"]["title"])
    
    return retrieve_docs, sources

In [9]:
def build_prompt(query, docs):
    prompt_start = "Answer the questionn using the context below:\n\n"
    context = "\n\n".join(docs)
    prompt_end = f"\n\nQuestion: {query}"
    return prompt_start + context + prompt_end

In [10]:
def question_answering(prompt, sources):
    response = client.chat.completions.create(
        model = "gpt-4o",
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]
    )
    
    answer = response.choices[0].message.content
    return{"answer": answer, "sources": sources}

In [11]:
query = "What is machine learning?"

docs, sources = retrieve_docs(query)

prompt = build_prompt(query, docs)

result = question_answering(prompt, sources)

print("Answer:\n", result["answer"])
print("\nSources:\n", result["sources"])

NameError: name 'client' is not defined